In [ ]:
%load_ext autoreload
%autoreload 2

import pyomo.environ as pyo
import numpy as np
import scipy.sparse as sp
from src.plotting import *
from src.traffic_sim import *
from src.mpc_metanet import *
import pandas as pd

import time
import json
import pyomo.contrib.parmest.utils.ipopt_solver_wrapper as ipopt_solver_wrapper
    

In [ ]:
# Test METANET simulation
import matplotlib.pyplot as plt
pred_horizon = 40
control_horizon = 5
sim_time = 720
time_step = 10/3600
L = 400/1000
time_steps = sim_time + pred_horizon - control_horizon
oscillating = False # Set oscillating to false if want to test a spike in demand instead of stop and go behavior
i24_data = False

if oscillating:
    step_per_min = int(60/3600 / (time_step))
    pattern = [0] * 30* step_per_min + [p_crit + 30] * 12 * step_per_min 
    repeat_count = 4 #(time_steps + 1 + len(pattern) - 1) // len(pattern)
    downstream_density = np.array(pattern * repeat_count + [0 for i in range(time_steps)])[:time_steps+1]

    traffic_demand = [4400 for i in range(time_steps + 1)]
elif i24_data:
    downstream_density = np.load('i24_data/downstream_density_400.npy').reshape(-1)
    traffic_demand = np.load('i24_data/initial_flow_400.npy').reshape(-1)

    sim_time = downstream_density.shape[0]
    time_steps = sim_time + pred_horizon - control_horizon
    while len(downstream_density) < time_steps + 1:
        downstream_density = np.append(downstream_density, downstream_density[-1])
        traffic_demand = np.append(traffic_demand, traffic_demand[-1])
else:
    downstream_density = np.full(time_steps + 1, 0)
    traffic_demand = [5500 if i in range(90, 90 + 180) else 4000 for i in range(time_steps + 1)]
    #traffic_demand = [2700 if i in range(50, 50 + 360) else 1000 for i in range(time_steps + 1)]
    # for i in range(150 + 270, 150 + 270 + 270):
    #     traffic_demand[i] = 2400

    # for i in range(100, 200):
    #     downstream_density[i] = p_crit * 0.9

    # traffic_demand = [3200 if i in range(150, time_steps+1) else 3200 for i in range(time_steps + 1)]


fig, axs = plt.subplots(1, 2, figsize=(15, 5))

axs[0].plot(downstream_density)
axs[0].set_title('Downstream Density')
# axs[0].set_xticks(np.arange(0, time_steps+1, 120), np.arange(0, int(time_steps/12)+1, int(120/12)))
axs[0].set_xlabel('Timesteps (10 sec)')
axs[0].set_ylabel('Density (veh/km)')

time = np.arange(0, time_steps + 1) * time_step
axs[1].plot(time, traffic_demand, linewidth=3)
axs[1].set_title('Traffic Demand', fontsize=20, fontname='Times New Roman')
axs[1].set_xlabel('Time (hrs)', fontsize=20, fontname='Times New Roman')
axs[1].set_ylabel('Total Inflow (veh/hr)', fontsize=20, fontname='Times New Roman')

for ax in axs:
    ax.tick_params(axis='both', which='major', labelsize=14)

plt.tight_layout()
plt.show()

In [ ]:
total_distance = 6
num_segments = int(total_distance/L)